# MMS_2D — method-of-manufactured-solutions forms, merged (2D)

`MagneticStandalone` · ASN 28 validation form (TRUST-LiMe).

Merged form (notebook-merging pass M4): the four historical 2D MMS
notebooks — `MMS_Diff2D`, `MMS_Trig2D`, `MMS_ExpTrig2D`, `MMS_Poly_2D` —
run as four datasets of a single form folder, each keeping its own
baseline case, checks and mesh-convergence scan. The datasets are
byte-identical to the historical ones (same names); their contracts are
unchanged.

| Case | Contract |
|---|---|
| `MMS_Diff2D` | uniform (0,1) field invariant under convection (machine precision) |
| `MMS_Trig2D` | uniform (0,1) field invariant under convection (machine precision) |
| `MMS_ExpTrig2D` | sin(pi x) exp(-y) decays at the local rate eta(pi^2 - 1) (±10%) |
| `MMS_Poly_2D` | zero field + zero source stays zero (guarded smoke check) |

Every case carries the shared refinement ladder (cells ×2 / ×4) on its
own template `src/<case>.tpl.data`.

In [ ]:
from trustutils import run
import numpy as np
import os
from trustutils.files import SonFile
import pathlib, shutil

run.TRUST_parameters("1.9.8")
run.reset()

NB_PROCS = 2
BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    cands = [f"{BUILD}/coarse/{case}_{name}.son",
             f"{BUILD}/{case}_{name}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{name}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_dir(subdir, case_name, name):
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)
print("BUILD =", BUILD)

In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder, per-case subdirectories) ---------
# Each single dataset src/<Case>.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<Case>/<scale>/ —
# one data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
N0 = 21                        # MMS_Diff2D / Trig2D / ExpTrig2D baseline mesh
NP0 = 33                       # MMS_Poly_2D baseline mesh
LEVELS  = {"coarse": N0, "standard": 2*N0-1, "fine": 4*N0-3}
LEVELSP = {"coarse": NP0, "standard": 2*NP0-1, "fine": 4*NP0-3}
cases = {}
for base, lv in (("MMS_Diff2D", LEVELS),
                 ("MMS_ExpTrig2D", LEVELS),
                 ("MMS_Poly_2D", LEVELSP),
                 ("MMS_Trig2D", LEVELS)):
    cases[base] = {}
    for lvl, nn in lv.items():
        cases[base][lvl] = run.addCaseFromTemplate(
            f"{base}.data", f"{base}/{lvl}", dic={"nx": nn, "ny": nn},
            nbProcs=NB_PROCS, excluNR=True)
        cases[base][lvl].partition()   # trust -partition <Case>.data 2 -> PAR_<Case>.data + Zones
run.printCases()
run.runCases()
c_diff = cases["MMS_Diff2D"]["coarse"]
c_trig = cases["MMS_Trig2D"]["coarse"]
c_exptrig = cases["MMS_ExpTrig2D"]["coarse"]
c_poly = cases["MMS_Poly_2D"]["coarse"]


## MMS_Diff2D — uniform (0,1) field invariance (convection on)

In [ ]:
t, bx = read_son(c_diff.dataFileName_, 'SONDE_BX')
_, by = read_son(c_diff.dataFileName_, 'SONDE_BY')
err_bx, err_by = np.max(np.abs(bx)), np.max(np.abs(by - 1.0))
assert err_bx < 1e-10 and err_by < 1e-10, f"MMS_Diff2D not invariant: {err_bx:.3e} {err_by:.3e}"
print(f"OK: MMS_Diff2D uniform (0,1) field invariant ({err_bx:.1e}, {err_by:.1e}).")

## MMS_Trig2D → MMS_Trig2D — uniform (0,1) field invariance

In [ ]:
t, bx = read_son(c_trig.dataFileName_, 'SONDE_BX')
_, by = read_son(c_trig.dataFileName_, 'SONDE_BY')
err_bx, err_by = np.max(np.abs(bx)), np.max(np.abs(by - 1.0))
assert err_bx < 1e-10 and err_by < 1e-10, f"MMS_Trig2D not invariant: {err_bx:.3e} {err_by:.3e}"
print(f"OK: MMS_Trig2D uniform (0,1) field invariant ({err_bx:.1e}, {err_by:.1e}).")

## MMS_ExpTrig2D — local-rate decay of sin(pi x) exp(-y)

In [ ]:
eta = 0.01
rate = eta * (np.pi**2 - 1.0)
for name in ('SONDE_Y025', 'SONDE_Y050', 'SONDE_Y075'):
    t, b = read_son(c_exptrig.dataFileName_, name)
    ratio = b[-1] / b[0]
    expected = np.exp(-rate * t[-1])
    err = abs(ratio - expected) / expected
    print(f"MMS_ExpTrig2D {name}: ratio={ratio:.4f} expected={expected:.4f} err={err:.3e}")
    assert err < 0.1, f"{name} decay error too large: {err:.3e}"
print("OK: MMS_ExpTrig2D exp-trig decay matches the local-rate prediction.")

## MMS_Poly_2D — zero field stays zero (guarded smoke check)

In [ ]:
try:
    t, bx = read_son(c_poly.dataFileName_, 'SONDE_BX')
    _, by = read_son(c_poly.dataFileName_, 'SONDE_BY')
    assert np.max(np.abs(bx)) < 1e-10 and np.max(np.abs(by)) < 1e-10, "poly_2D not zero"
    print("OK: MMS_Poly_2D field stays near zero.")
except Exception as e:
    print(f"Note: could not read MMS_Poly_2D probes: {e}")
    print("(the baseline dataset is probeless; the identity is asserted at the scan levels below)")

## Mesh convergence (all four cases)

Shared refinement ladder per case (cells ×2 / ×4 from each baseline
mesh), instantiated from the case's own `src/<case>.tpl.data`; each level
runs in `build/<case>/<scale>/` so the partitioned Zones never
collide. Contracts: the invariance cases (Diff2D, Trig2D) and the
zero-field case (Poly2D, via template-only probes at the scan levels)
assert their machine-precision identity at every level; ExpTrig2D uses
the exact-implicit-Euler-cancelled decay with the analytic local rate
lambda = eta(pi^2 - 1) to isolate the spatial error (monotone decrease,
observed order log2(e_k/e_{k+1})).

In [ ]:
def inv_level_error(sub, cn):
    _, bx = read_son_dir(sub, cn, "SONDE_BX")
    _, by = read_son_dir(sub, cn, "SONDE_BY")
    return np.max(np.abs(bx)), np.max(np.abs(by - 1.0))

for base, cvar in (("MMS_Diff2D", c_diff), ("MMS_Trig2D", c_trig)):
    print(f"-- {base} (21/41/81)")
    err = {"coarse": inv_level_error(f"{base}/coarse", cvar.dataFileName_)}
    for lvl, nn in LEVELS.items():
        err[lvl] = inv_level_error(f"{base}/{lvl}", cases[base][lvl].dataFileName_)
    for lvl in ("coarse", "standard", "fine"):
        ex, ey = err[lvl]
        print(f"   {lvl}: |Bx|={ex:.1e} |By-1|={ey:.1e}")
        assert ex < 1e-10 and ey < 1e-10, f"{base} identity broken at {lvl}"
    print(f"OK: {base} invariance holds at every level.")

In [ ]:
def zero_level_error(sub, cn):
    _, bx = read_son_dir(sub, cn, "SONDE_BX")
    _, by = read_son_dir(sub, cn, "SONDE_BY")
    return max(np.max(np.abs(bx)), np.max(np.abs(by)))

print("-- MMS_Poly_2D (33/65/129, template-only probes at the scan levels)")
for lvl, nn in LEVELSP.items():
    e = zero_level_error(f"MMS_Poly_2D/{lvl}", cases["MMS_Poly_2D"][lvl].dataFileName_)
    print(f"   {lvl}: max|b|={e:.1e}")
    assert e < 1e-10, f"MMS_Poly_2D zero-field identity broken at {lvl}"
print("OK: MMS_Poly_2D zero-field identity holds at every instrumented level.")

In [ ]:
SONDE_TAGS = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
ETA = 0.01
LAM = ETA * (np.pi**2 - 1.0)

def implicit_ref(t, lam):
    t = np.asarray(t, dtype=float)
    ref = np.ones_like(t)
    for i in range(1, len(t)):
        ref[i:] *= 1.0 / (1.0 + lam * (t[i] - t[i-1]))
    return ref

def spatial_ratio_error(sub, cn):
    worst = 0.0
    for tag in SONDE_TAGS:
        t, b = read_son_dir(sub, cn, tag)
        if b[0] == 0: continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - implicit_ref(t, LAM)))))
    return worst

print("-- MMS_ExpTrig2D (21/41/81): |ratio - implicit-Euler ref|")
base = "MMS_ExpTrig2D"
err = {"coarse": spatial_ratio_error(f"{base}/coarse", c_exptrig.dataFileName_)}
for lvl in LEVELS:
    err[lvl] = spatial_ratio_error(f"{base}/{lvl}", cases[base][lvl].dataFileName_)
prev = None
for lvl in ("coarse", "standard", "fine"):
    e = err[lvl]
    order = "--" if prev is None else f"{np.log2(prev/e):.2f}"
    print(f"   {lvl}: {e:.4e}  order {order}")
    if prev is not None:
        assert e < prev, f"MMS_ExpTrig2D no convergence {err}"
    prev = e
print("OK: MMS_ExpTrig2D spatial error decreases under refinement.")

## Computer Performance

In [ ]:
run.tablePerf()